# Four faults, one symptom

MichAl Academy, lesson 2.12.

Run each cell with **Shift+Enter**.

A model that is wrong is one observation with several possible causes, and the
causes need completely different responses. Collect more data, fix a threshold,
delete a column, or start again on the features: pick wrong and you spend a
month on the wrong thing.

So this notebook breaks four models on purpose and runs the same five checks
against all of them. What matters is not that each check is true. It is which
checks stay **silent** for the other three faults, because that is what makes a
check diagnostic rather than merely informative.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold)
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score

SEED = 0
bc = load_breast_cancer()
X0, y0 = bc.data, bc.target
names0 = list(bc.feature_names)
print(f"the healthy starting point: {len(y0)} rows, {X0.shape[1]} features")


## The five checks

Written once, run against everything. Read the code, because four of the five
are two lines and the fifth is the one worth learning.


In [ ]:
def diagnose(label, split, names, dataset):
    X_tr, X_te, y_tr, y_te = split
    model = RandomForestClassifier(random_state=SEED, n_jobs=1).fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]

    # 1. Is there any lift over answering with the majority class?
    acc = accuracy_score(y_te, pred)
    majority = max(y_te.mean(), 1 - y_te.mean())

    # 2. Can it rank at all, separately from where the threshold sits?
    auc = roc_auc_score(y_te, proba)
    rec = recall_score(y_te, pred, zero_division=0)
    caught = int(((pred == 1) & (y_te == 1)).sum())
    real = int((y_te == 1).sum())

    # 3. Is one feature doing everything? Both the magnitude and the share,
    #    because neither is diagnostic without the other.
    pi = permutation_importance(model, X_te, y_te, n_repeats=5,
                                random_state=SEED, n_jobs=1)
    imp = pi.importances_mean
    top = int(np.argmax(imp))
    positive = imp[imp > 0]

    # 4. Adversarial validation. Label every training row 0 and every test row
    #    1, then ask a model to tell them apart. If it can, they are not
    #    samples of the same population and nothing else you measure is safe.
    Xa = np.vstack([X_tr, X_te])
    ya = np.concatenate([np.zeros(len(X_tr)), np.ones(len(X_te))])
    adv = cross_val_score(
        RandomForestClassifier(random_state=SEED, n_jobs=1), Xa, ya,
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc"
    ).mean()

    # 5. Drop the top feature and refit. Cheap, and it confirms check 3.
    keep = [k for k in range(X_tr.shape[1]) if k != top]
    without = RandomForestClassifier(random_state=SEED, n_jobs=1) \
        .fit(X_tr[:, keep], y_tr).score(X_te[:, keep], y_te)

    print(f"\n{label}   [{dataset}]")
    print(f"  1. accuracy {acc:.4f}   majority baseline {majority:.4f}"
          f"   lift {acc - majority:+.4f}")
    print(f"  2. ROC-AUC {auc:.4f}   recall {rec:.4f}"
          f"   (caught {caught} of {real})")
    if positive.size:
        print(f"  3. top feature '{names[top]}' importance {imp[top]:+.4f},"
              f" {imp[top] / positive.sum():.1%} of the positive total")
    else:
        print(f"  3. no feature had a positive importance at all")
    print(f"  4. train against test, adversarial ROC-AUC {adv:.4f}")
    print(f"  5. accuracy without that feature {without:.4f}"
          f"   change {without - acc:+.4f}")


healthy = train_test_split(X0, y0, test_size=0.3, random_state=SEED, stratify=y0)
diagnose("HEALTHY", healthy, names0, "breast cancer")


That is the reference reading. Big lift, high ROC-AUC, no feature above 14%,
and adversarial validation at 0.46, which is chance. All five checks quiet.

Everything below is compared against those numbers.


## Fault 1: leakage

A column computed from the label. In real work it does not arrive labelled
`leak`, it arrives as `review_outcome` or `risk_tier` or a field somebody filled
in after the answer was known. Lesson 2.3 is about how it gets in; this is about
spotting it once it has.


In [ ]:
review = y0 * 0.9 + np.random.default_rng(SEED).normal(0, 0.15, size=len(y0))
X_leak = np.column_stack([X0, review])
diagnose("LEAKAGE",
         train_test_split(X_leak, y0, test_size=0.3, random_state=SEED, stratify=y0),
         names0 + ["review outcome"], "breast cancer plus one column")


Two readings give it away and neither is subtle.

**Accuracy of exactly 1.0000.** Not 0.997. Perfect. Real problems are not
perfect, and a perfect score is a bug report.

**One feature at 97.8% of the importance, with an importance of +0.2561**, which
is twelve times the healthy model's top feature.

And check 5 closes it. Drop that column and accuracy is 0.9532, which is the
healthy model's score to four decimals. The other thirty features were doing
their job the whole time; the leak was adding nothing but the answer.


## Fault 2: class imbalance

Breast cancer cannot be made severely imbalanced and still leave enough
positives to read a recall from, so this fault borrows lesson 2.7's setup:
digits, is it an 8, keeping one in ten of the 8s.


In [ ]:
dg = load_digits()
y8 = (dg.target == 8).astype(int)
neg, pos = np.flatnonzero(y8 == 0), np.flatnonzero(y8 == 1)
keep = np.concatenate([neg, np.random.default_rng(SEED).choice(
    pos, size=len(pos) // 10, replace=False)])
X_imb, y_imb = dg.data[keep], y8[keep]
print(f"{len(y_imb)} rows, {y_imb.sum()} positives, base rate {y_imb.mean():.4f}")

diagnose("IMBALANCE",
         train_test_split(X_imb, y_imb, test_size=0.3, random_state=SEED, stratify=y_imb),
         [f"pixel {i}" for i in range(X_imb.shape[1])], "digits, is it an 8")


**Lift of exactly +0.0000.** Accuracy 0.9898, and answering "no" to everything
scores 0.9898. To four decimal places. That is the signature.

**And ROC-AUC 0.7838**, comfortably above the 0.5 of a coin toss. Which is the
important part: the model **can** rank. What it cannot do is decide, because 0.5
is the wrong place to cut. This is not a model problem, it is a threshold
problem, and lesson 2.8 is the fix rather than more data or a bigger model.

Check 3 has its own tell here: **no feature had a positive importance at all.**
Permuting a column cannot change the accuracy of a model that answers "no" to
everything, so nothing looks important because nothing is being used.


## Fault 3: distribution shift

Sort the rows by mean radius, train on the smaller 70%, test on the larger 30%.
Nothing is fabricated. The rows are real and the split is simply not random,
which is what happens when your training data is last year and your test data
is this year.


In [ ]:
j = names0.index("mean radius")
order = np.argsort(X0[:, j])
cut = int(len(order) * 0.7)
tr, te = order[:cut], order[cut:]
diagnose("SHIFT", (X0[tr], X0[te], y0[tr], y0[te]), names0, "breast cancer, sorted split")


Now read all five, in order, and notice what happens.

Accuracy 0.9415, which on its own looks healthy. ROC-AUC 0.9699, healthy. Top
feature at 17%, healthy. Dropping it costs a point, healthy.

**And adversarial validation is 0.9999.** A model can tell a training row from a
test row essentially every time.

Four checks silent, one screaming. That is the whole argument for learning check
4, because this is the fault most likely to be sitting in a system that looks
fine, and the only cheap way to find it is to stop asking about the label for a
moment and ask about the split.

The 0.0058 lift is the second clue, but you only see it if you were already
printing the baseline, and 0.9415 accuracy on its own does not invite suspicion.


## Fault 4: no signal

Replace every feature with noise of the same shape. The labels are untouched and
real.


In [ ]:
X_noise = np.random.default_rng(SEED).normal(size=X0.shape)
diagnose("NO SIGNAL",
         train_test_split(X_noise, y0, test_size=0.3, random_state=SEED, stratify=y0),
         names0, "breast cancer, columns replaced")


**ROC-AUC 0.4622.** A coin toss, and slightly worse than one. There is no
ranking to be had, so nothing the model was given relates to the label.

Now look at check 3, because it contains a trap worth more than the rest of this
notebook. The top feature holds **75.0% of the positive total**, which is the
leakage signature. But its importance is **+0.0035**, against leakage's +0.2561.

Seventy-three times smaller. What happened is that with noise features almost
every permutation importance comes out negative, so whichever one lands
slightly positive takes most of a denominator that means nothing.

**A share is not a measurement when the denominator is noise.** Read the
magnitude beside it or check 3 will send you looking for a leak in a dataset
that has no signal at all.


## The table

Every reading, side by side.

```
                 lift   ROC-AUC   top importance    share    adversarial
healthy       +0.3275    0.9781          +0.0211    13.7%         0.4613
leakage       +0.3743    1.0000          +0.2561    97.8%         0.4882
imbalance     +0.0000    0.7838          +0.0000     none         0.4890
shift         +0.0058    0.9699          +0.0094    17.0%         0.9999
no signal     -0.0643    0.4622          +0.0035    75.0%         0.4638
```

And the decision procedure it supports, in the order worth running it:

1. **Adversarial ROC-AUC well above 0.5?** Distribution shift. Fix the split or
   the sampling before measuring anything else, because every other number is
   answering a question about a population you are not deploying to.
2. **Score suspiciously perfect, one feature large and dominant?** Leakage.
   Drop it and see whether the score returns to something believable.
3. **Lift near zero but ROC-AUC well above 0.5?** Imbalance or a threshold left
   at 0.5. Lesson 2.8, not more data.
4. **ROC-AUC near 0.5?** No signal. The problem is in the features or the
   labels, and no model or amount of tuning will reach it.

Run it in that order because the earlier faults invalidate the later checks. A
shifted split makes every other reading a measurement of the wrong thing.


## What to take from this

| Claim | What we measured |
|---|---|
| A bad score tells you the model needs work | It has at least four causes needing four different responses |
| High accuracy means the model is fine | The shifted model scored 0.9415 and was trained on a different population |
| One dominant feature means leakage | Not on its own. The no-signal model showed 75.0% with an importance of +0.0035 |
| Poor recall means you need more data | Not if ROC-AUC is high. Then it is a threshold, and 2.8 fixes it for free |
| Adversarial validation is an exotic technique | Four lines, and it was the only check that saw the shift |

The habit: **print the majority baseline and the adversarial AUC beside every
score you report.** Both are two lines, both are silent when things are fine,
and between them they catch the two faults that look like success.


## Try this

1. Make the shift milder: split at the 40th percentile of mean radius instead
   of sorting completely. How high does the adversarial AUC stay, and at what
   point does it stop being obvious?
2. Set `review = y0 * 0.3 + noise * 0.7` in fault 1, a weak leak rather than a
   blatant one. Does check 3 still find it? This is the realistic case.
3. Run all five checks on your own most recent model. The point of the notebook
   is that the function at the top is reusable, and takes about a minute.
